# 14. ComfyUI: от визуального baseline к отдельному исследованию

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 1](https://diffusion.csail.mit.edu/2026/docs/20260120_Lecture_01.pdf) · [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf) · [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf) · [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** связать UI nodes с кодовыми компонентами и подготовить измеряемые однородные A/B опыты

**Перед началом:** K6 01–13; ComfyUI C0–C5 общего маршрута. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/10_colab_cuda_distill.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §5–6](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)
- [`core/algo/piflow_sampler.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/piflow_sampler.py)
- [`core/algo/postprocess_video.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/postprocess_video.py)
- [`core/algo/postprocess_audio.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/postprocess_audio.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '14_comfyui_component_experiments'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. UI показывает wiring, а не все внутренние слои

Loader DiT подключает joint transformer; DualCLIP loader — Qwen+CLIP; VAE video и audio-vocoder
обслуживают разные decoding paths. Один sampler node скрывает много CrossDiT forwards и policy updates.
Внутренние gates нельзя честно «выключить» простым отсоединением audio decoder:
decoder находится **после** совместного denoising.

Наши Smoke/Baseline workflows используют Lite Distill без SR/beautifier/MagCache.
Node names/widgets зависят от версии ComfyUI и extension. Перед исполнением сверяйте template/node implementation.
Следующая ячейка только читает локальный JSON; в ComfyUI ничего не отправляет.


In [ ]:
workflow=json.loads((ROOT/'workflows/k6_lite_distill_smoke.json').read_text())
rows=[{'id':n['id'],'type':n['type'],'title':n.get('title',n['type']),
       'widgets':n.get('widgets_values',[])} for n in workflow['nodes']]
for row in rows:print(row)
metrics['smoke_node_count']=len(rows);metrics['smoke_link_count']=len(workflow['links'])
assert len(rows)==12 and len(workflow['links'])==14


## 2. Последовательность опытов и измерений

| Опыт | Что меняем | Что фиксируем | Что измеряем |
|---|---|---|---|
| C0 | Только image IO без моделей | PNG размеры/цвета | exact roundtrip / MSE |
| C4 | Video VAE encode→decode | input, codec scale, dtype | reconstruction MSE/PSNR, decode time |
| K6-S | Первый Lite Smoke | seed=42, 432×240, 33 frames, 10 steps | success, stage times, system memory |
| K6-B | Полный Baseline 5s | 864×480, 121 frames; те же weights | время/память, frames/audio samples |
| K6-A | Только один prompt factor | все остальные widgets/seeds | text adherence, failures, audio transcript |
| K6-R | Адаптер speaker condition | held-out speaker/text protocol | SECS/WER/CER, no-reference regression |

[C0/C4 workflows и инструкции](../../curriculum/COMFYUI_LABS.md),
[ручной K6 notebook](00_manual_experiment_comfy_mlx_film.ipynb),
[протокол измерений](../COMFYUI_AND_EXPERIMENT_RU.md).
K6-S и дальше выполняются **после выбранного SSD gate и проверки моделей**, вручную.
До этого таблицы содержат `not_run`, не выдуманные числа.


In [ ]:
import csv
folder=ROOT/'artifacts/component_labs'/LAB;folder.mkdir(parents=True,exist_ok=True)
columns=['run_id','experiment','checkpoint','workflow','seed','steps','width','height','frames',
         'load_s','text_s','denoise_s','video_decode_s','audio_decode_s','mux_s','peak_memory_bytes','status','notes']
path=folder/'manual_runs.csv'
if not path.exists():
    with path.open('w',newline='') as handle:
        writer=csv.DictWriter(handle,fieldnames=columns);writer.writeheader()
        writer.writerow({'run_id':'smoke-42','experiment':'K6-S','checkpoint':'Lite-distill-5s',
             'workflow':'k6_lite_distill_smoke.json','seed':42,'steps':10,'width':432,'height':240,
             'frames':33,'status':'not_run','notes':'Await SSD/model gate; empty metrics are not zero'})
print('Локальная таблица:',path)
metrics['prepared_experiments']=['C0','C1','C4','K6-S','K6-B','K6-A','K6-R']
metrics['comfyui_jobs_submitted']=0


## 3. Как перейти к исследованию голоса

Для адаптера нужны прочитанные codec/conditioning/fusion contracts, baseline исходной модели,
разделение VCTK по текстам и схема insertion point. Zero-init должен сохранять исходную функцию
на шаге 0; frozen backbone не означает запрет gradient через него к адаптеру.

[Наш исследовательский протокол](../../curriculum/RESEARCH_PROTOCOL.md) и
[K6 status](../STATUS.md) перечисляют незавершённые части. Статья о voice adaptation
[arXiv:2608.15690](https://arxiv.org/abs/2608.15690) обучает также audio backbone;
наш frozen-FiLM вариант — отдельная гипотеза. Toy sensitivity из notebook 08 помогает проверить
wiring, но не заменяет SECS/WER и не подтверждает voice cloning.


## Самостоятельное исследование

1. На скриншоте workflow подпишите, где text encoder, DiT, policy, video codec, mel codec и vocoder.
2. Составьте A/B из 3 одинаковых seeds и одного изменённого prompt factor; заранее определите метрики.
3. Почему выключение mux audio input не является audio-stream ablation внутри CrossDiT?
4. Укажите stop rule: ресурсный лимит, недостоверная метрика или regression исходной модели.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

У вас есть маршрут от формулы и native CPU опыта к UI baseline и отдельной исследовательской гипотезе,
с явным разделением выполненных численных контролей и будущих pretrained замеров.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 13. Runtime: память, offload, cache и путь к Apple MLX](13_runtime_memory_and_mlx.ipynb) · [Далее: отдельное исследование →](00_manual_experiment_comfy_mlx_film.ipynb)
